# Data and Mode-C setup


In [11]:
from pathlib import Path
import importlib.util
import warnings
import numpy as np
from scipy.optimize import differential_evolution, minimize


def _load_transfer_matrices_module():
    candidates = [
        Path.cwd() / "src" / "temgym_core" / "transfer_matrices.py",
        (Path.cwd() / "../../src/temgym_core/transfer_matrices.py").resolve(),
    ]
    for path in candidates:
        if path.exists():
            spec = importlib.util.spec_from_file_location("transfer_matrices_local", path)
            module = importlib.util.module_from_spec(spec)
            assert spec.loader is not None
            spec.loader.exec_module(module)
            return module
    raise FileNotFoundError("Could not find transfer_matrices.py in expected locations.")


_tm = _load_transfer_matrices_module()
propagation_matrix = _tm.propagation_matrix
lens_matrix = _tm.lens_matrix

# ------------------------------------------------------------------
# Centralized configuration
# ------------------------------------------------------------------
MODE_LABEL = "C"
M_obj = 60.0
M_proj = 100.0

I_fs_common_A = 2.0
N_TURNS = {
    "IL1": 2950,
    "IL2": 2050,
    "IL3": 2000,
    "OLf": 2000,
    "PL1": 2650,
}

RC_RAD_PER_AT = 5.3e-4
psi_target_rms_deg = 0.5

dpl1_norm_min = -0.020
dpl1_norm_max = +0.020

w_ni_grid = [0.0, 0.03, 0.1, 0.3, 1.0, 3.0]

# ------------------------------------------------------------------
# RAW DAC data - JEOL ARM-200F magnification lookup table
# ------------------------------------------------------------------
RAW_DAC_HEX = {
    "2000":    {"IL1": "0x4b4b", "IL2": "0x4b2b", "IL3": "0xaf98", "OLf": "0x625a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "2500":    {"IL1": "0x4cd7", "IL2": "0x458f", "IL3": "0xb471", "OLf": "0x562a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "3000":    {"IL1": "0x4e0b", "IL2": "0x4046", "IL3": "0xb8fd", "OLf": "0x582a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "4000":    {"IL1": "0x4fcf", "IL2": "0x3657", "IL3": "0xc0d7", "OLf": "0x656a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "5000":    {"IL1": "0x512f", "IL2": "0x2d1d", "IL3": "0xc719", "OLf": "0x6d6a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "6000":    {"IL1": "0x526e", "IL2": "0x2515", "IL3": "0xcccc", "OLf": "0x684a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "8000":    {"IL1": "0x6a31", "IL2": "0x607f", "IL3": "0xadd0", "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xe2cb"},
    "10000":   {"IL1": "0x6fe4", "IL2": "0x5c11", "IL3": "0xb1c7", "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xe5a6"},
    "12000":   {"IL1": "0x73f5", "IL2": "0x5925", "IL3": "0xb6fe", "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xeb2f"},
    "15000":   {"IL1": "0x7830", "IL2": "0x56ad", "IL3": "0xc026", "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xf548"},
    "20000":   {"IL1": "0x8057", "IL2": "0x52d6", "IL3": "0xbeb9", "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xf9bb"},
    "25000":   {"IL1": "0x8692", "IL2": "0x50e2", "IL3": "0x9b35", "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xffff"},
    "30000":   {"IL1": "0x9f22", "IL2": "0x611e", "IL3": "0x9b35", "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "40000":   {"IL1": "0xa336", "IL2": "0x611e", "IL3": "0x8ad8", "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "50000":   {"IL1": "0xa674", "IL2": "0x6498", "IL3": "0x85b7", "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "60000":   {"IL1": "0xa810", "IL2": "0x6a48", "IL3": "0x7f59", "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "80000":   {"IL1": "0xa8ea", "IL2": "0x7182", "IL3": "0x77d2", "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "100000":  {"IL1": "0xa800", "IL2": "0x7d46", "IL3": "0x7205", "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "120000":  {"IL1": "0xa7d4", "IL2": "0x7d46", "IL3": "0x6c7f", "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "150000":  {"IL1": "0xa761", "IL2": "0x8431", "IL3": "0x65e6", "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "200000":  {"IL1": "0xa6c5", "IL2": "0x957b", "IL3": "0x5c4a", "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "250000":  {"IL1": "0xa6a8", "IL2": "0x957b", "IL3": "0x552b", "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "300000":  {"IL1": "0xa608", "IL2": "0x9ec6", "IL3": "0x4ba8", "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "400000":  {"IL1": "0xa584", "IL2": "0xad0c", "IL3": "0x3e16", "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "500000":  {"IL1": "0xa51d", "IL2": "0xb944", "IL3": "0x3268", "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "600000":  {"IL1": "0xa4d0", "IL2": "0xc3c7", "IL3": "0x2771", "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "800000":  {"IL1": "0xe300", "IL2": "0xe54f", "IL3": "0x478",  "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "1000000": {"IL1": "0xe300", "IL2": "0xbd3a", "IL3": "0x1ac9", "OLf": "0x93ca", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "1200000": {"IL1": "0xe300", "IL2": "0xcd09", "IL3": "0x1ac9", "OLf": "0x93ca", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "1500000": {"IL1": "0xe300", "IL2": "0xec00", "IL3": "0x0",   "OLf": "0x93ca", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "2000000": {"IL1": "0xffff", "IL2": "0xf800", "IL3": "0x0",   "OLf": "0x93ca", "OLc": "0xb2fa", "PL1": "0xfa00"},
}

LENS_NAMES = ["IL1", "IL2", "IL3", "PL1"]
AUX_NAMES = ["OLf", "OLc"]


def parse_dac_table(raw):
    """Convert hex DAC table to structured arrays and operating-mode labels."""
    mags = sorted(int(k) for k in raw.keys())
    data = {"mag": np.array(mags, dtype=float)}

    for lens in LENS_NAMES + AUX_NAMES:
        data[lens] = np.array([int(raw[str(m)][lens], 16) for m in mags], dtype=float)

    modes = []
    for m in mags:
        if m <= 6000:
            modes.append("A")
        elif m <= 25000:
            modes.append("B")
        elif m <= 600000:
            modes.append("C")
        else:
            modes.append("D")
    data["mode"] = np.array(modes)
    return data


dac = parse_dac_table(RAW_DAC_HEX)
mode_idx = np.where(dac["mode"] == MODE_LABEL)[0]
mode_mag = dac["mag"][mode_idx]
MILSystem = mode_mag / (M_obj * M_proj)

# Nominal geometry
z2_proj = 325e-3
z1_proj = z2_proj / M_proj

d_il1_to_il2_nom = 52e-3
d_il2_to_il3_nom = 62e-3
d_il3_to_im_nom = 52e-3 - z1_proj

print(f"Mode {MODE_LABEL} settings: {len(mode_idx)}")
print(f"Target intermediate magnification range: {MILSystem.min():.2f} .. {MILSystem.max():.2f}")




Mode C settings: 14
Target intermediate magnification range: 5.00 .. 100.00


# Optical fit


In [12]:
# Optical-model helpers (IL1/IL2/IL3 fit only)

def lens_f_from_current(I_norm, G_c):
    power = G_c * I_norm**2
    if power <= 0:
        return 1e9
    return 1.0 / power


def lens_thickness_from_current(I_norm, C_D):
    # D(I) = C_D * I_norm, clipped to avoid negative thickness.
    return max(C_D * I_norm, 0.0)


def thick_lens_matrix_from_current(I_norm, G_c, C_D):
    """
    Symmetric thick-lens model:
        L_back(2f) @ P(D) @ L_front(2f)
    with f = 1 / (G_c * I_norm^2), D = C_D * I_norm.
    """
    f = lens_f_from_current(I_norm, G_c)
    D = lens_thickness_from_current(I_norm, C_D)

    l_front = lens_matrix(2.0 * f, xp=np)
    p_mid = propagation_matrix(D, xp=np)
    l_back = lens_matrix(2.0 * f, xp=np)
    return l_back @ p_mid @ l_front


def build_loss(dac, mode_idx, target_mags):
    def loss(params):
        (
            d_obj_to_il1,
            d_il1_to_il2,
            d_il2_to_il3,
            d_il3_to_im,
            G_c_1,
            C_D_1,
            G_c_2,
            C_D_2,
            G_c_3,
            C_D_3,
        ) = params

        total_loss = 0.0
        predicted = []

        for i, mag_target in zip(mode_idx, target_mags):
            I1 = dac["IL1"][i] / 2**16
            I2 = dac["IL2"][i] / 2**16
            I3 = dac["IL3"][i] / 2**16

            p1 = propagation_matrix(d_obj_to_il1, xp=np)
            p2 = propagation_matrix(d_il1_to_il2, xp=np)
            p3 = propagation_matrix(d_il2_to_il3, xp=np)
            p4 = propagation_matrix(d_il3_to_im, xp=np)

            l1 = thick_lens_matrix_from_current(I1, G_c_1, C_D_1)
            l2 = thick_lens_matrix_from_current(I2, G_c_2, C_D_2)
            l3 = thick_lens_matrix_from_current(I3, G_c_3, C_D_3)

            M_total = p4 @ l3 @ p3 @ l2 @ p2 @ l1 @ p1
            A = float(M_total[0, 0])
            B = float(M_total[0, 1])

            current_mag = abs(A)
            predicted.append(current_mag)

            # Primary objective: focus (B ~ 0) + magnification match.
            total_loss += (B * 1000.0) ** 2 * 10.0
            total_loss += ((current_mag - mag_target) / mag_target) ** 2 * 10000.0

        # Soft monotonicity penalty.
        predicted = np.array(predicted)
        diffs = np.diff(predicted)
        violations = diffs[diffs <= 0.0]
        if len(violations) > 0:
            total_loss += np.sum(violations**2) * 1e6

        return total_loss

    return loss


loss = build_loss(dac=dac, mode_idx=mode_idx, target_mags=MILSystem)

bounds = [
    (8e-3, 20e-3),
    (d_il1_to_il2_nom - 2e-3, d_il1_to_il2_nom + 2e-3),
    (d_il2_to_il3_nom - 2e-3, d_il2_to_il3_nom + 2e-3),
    (d_il3_to_im_nom - 2e-3, d_il3_to_im_nom + 2e-3),
    (10.0, 5000.0), (0.0, 10e-3),
    (10.0, 5000.0), (0.0, 10e-3),
    (10.0, 5000.0), (0.0, 10e-3),
]


In [13]:
# Hybrid optimization: global search then local refinement.
result_de = differential_evolution(
    loss,
    bounds,
    maxiter=120,
    popsize=12,
    tol=1e-4,
    mutation=(0.5, 1.5),
    recombination=0.7,
    seed=42,
    polish=False,
)

result = minimize(loss, result_de.x, bounds=bounds, method="L-BFGS-B")

(
    d_obj_to_il1_opt,
    d_il1_to_il2_opt,
    d_il2_to_il3_opt,
    d_il3_to_im_opt,
    G_c_1_opt,
    C_D_1_opt,
    G_c_2_opt,
    C_D_2_opt,
    G_c_3_opt,
    C_D_3_opt,
) = result.x

if not result.success:
    warnings.warn(f"Optimization did not converge cleanly: {result.message}")

print("Fit summary:")
print(f"  success={result.success}, DE_loss={result_de.fun:.3f}, refined_loss={result.fun:.3f}")
print("  geometry [mm]:")
print(f"    d_obj_to_il1={d_obj_to_il1_opt*1e3:.3f}")
print(f"    d_il1_to_il2={d_il1_to_il2_opt*1e3:.3f} (nominal {d_il1_to_il2_nom*1e3:.3f})")
print(f"    d_il2_to_il3={d_il2_to_il3_opt*1e3:.3f} (nominal {d_il2_to_il3_nom*1e3:.3f})")
print(f"    d_il3_to_im ={d_il3_to_im_opt*1e3:.3f} (nominal {d_il3_to_im_nom*1e3:.3f})")
print("  lens params:")
print(f"    IL1: G_c={G_c_1_opt:.3f}, C_D={C_D_1_opt*1e3:.3f} mm/I_norm")
print(f"    IL2: G_c={G_c_2_opt:.3f}, C_D={C_D_2_opt*1e3:.3f} mm/I_norm")
print(f"    IL3: G_c={G_c_3_opt:.3f}, C_D={C_D_3_opt*1e3:.3f} mm/I_norm")


Fit summary:
  success=True, DE_loss=919.585, refined_loss=818.205
  geometry [mm]:
    d_obj_to_il1=8.000
    d_il1_to_il2=52.839 (nominal 52.000)
    d_il2_to_il3=60.000 (nominal 62.000)
    d_il3_to_im =50.750 (nominal 48.750)
  lens params:
    IL1: G_c=348.487, C_D=0.626 mm/I_norm
    IL2: G_c=314.790, C_D=3.593 mm/I_norm
    IL3: G_c=102.702, C_D=10.000 mm/I_norm


In [14]:
def summarize_fit_metrics(target_mag, actual_mag, b_mm):
    rel_err_pct = 100.0 * (actual_mag - target_mag) / target_mag
    return {
        "mag_rel_rmse_pct": float(np.sqrt(np.mean(rel_err_pct**2))),
        "mag_rel_max_abs_pct": float(np.max(np.abs(rel_err_pct))),
        "b_rms_mm": float(np.sqrt(np.mean(b_mm**2))),
        "b_max_abs_mm": float(np.max(np.abs(b_mm))),
        "monotonic_violations": int(np.sum(np.diff(actual_mag) <= 0.0)),
    }


actual_mag = []
b_mm = []
for i in mode_idx:
    I1 = dac["IL1"][i] / 2**16
    I2 = dac["IL2"][i] / 2**16
    I3 = dac["IL3"][i] / 2**16

    p1 = propagation_matrix(d_obj_to_il1_opt, xp=np)
    p2 = propagation_matrix(d_il1_to_il2_opt, xp=np)
    p3 = propagation_matrix(d_il2_to_il3_opt, xp=np)
    p4 = propagation_matrix(d_il3_to_im_opt, xp=np)

    l1 = thick_lens_matrix_from_current(I1, G_c_1_opt, C_D_1_opt)
    l2 = thick_lens_matrix_from_current(I2, G_c_2_opt, C_D_2_opt)
    l3 = thick_lens_matrix_from_current(I3, G_c_3_opt, C_D_3_opt)

    M_total = p4 @ l3 @ p3 @ l2 @ p2 @ l1 @ p1
    actual_mag.append(abs(float(M_total[0, 0])))
    b_mm.append(float(M_total[0, 1]) * 1e3)

actual_mag = np.array(actual_mag)
b_mm = np.array(b_mm)
fit_metrics = summarize_fit_metrics(target_mag=MILSystem, actual_mag=actual_mag, b_mm=b_mm)

print("Magnification/focus summary:")
print(
    f"  mag RMSE={fit_metrics['mag_rel_rmse_pct']:.3f}%"
    f", mag max abs={fit_metrics['mag_rel_max_abs_pct']:.3f}%"
    f", B_rms={fit_metrics['b_rms_mm']:.3f} mm"
    f", B_max_abs={fit_metrics['b_max_abs_mm']:.3f} mm"
)

print("\nCompact fit table:")
print(f"{'Mag target':>10s} | {'Mag actual':>10s} | {'Err %':>8s} | {'B [mm]':>9s}")
print("-" * 50)
for mag_t, mag_a, b in zip(MILSystem, actual_mag, b_mm):
    err = 100.0 * (mag_a - mag_t) / mag_t
    print(f"{mag_t:10.2f} | {mag_a:10.2f} | {err:8.3f} | {b:9.3f}")


Magnification/focus summary:
  mag RMSE=5.518%, mag max abs=12.703%, B_rms=1.673 mm, B_max_abs=3.224 mm

Compact fit table:
Mag target | Mag actual |    Err % |    B [mm]
--------------------------------------------------
      5.00 |       4.97 |   -0.677 |    -1.954
      6.67 |       6.37 |   -4.382 |    -3.224
      8.33 |       7.57 |   -9.116 |    -1.433
     10.00 |       9.62 |   -3.824 |    -0.260
     13.33 |      12.95 |   -2.846 |     1.228
     16.67 |      18.23 |    9.390 |     1.487
     20.00 |      19.70 |   -1.488 |     0.929
     25.00 |      24.68 |   -1.263 |     1.011
     33.33 |      37.57 |   12.703 |     2.538
     41.67 |      40.26 |   -3.378 |     2.052
     50.00 |      50.12 |    0.236 |     0.863
     66.67 |      66.51 |   -0.235 |     0.299
     83.33 |      81.35 |   -2.382 |    -0.882
    100.00 |      94.50 |   -5.496 |    -2.237


# NI-balance model


In [15]:
def compute_rotation_from_delta_ni(delta_ni_at, rc_rad_per_at=RC_RAD_PER_AT):
    return np.rad2deg(rc_rad_per_at * np.asarray(delta_ni_at, dtype=float))


def compute_ni_balance_terms(dac, mode_idx, turns, i_fs_common_a):
    i_norm = {
        "IL1": dac["IL1"][mode_idx] / 2**16,
        "IL2": dac["IL2"][mode_idx] / 2**16,
        "IL3": dac["IL3"][mode_idx] / 2**16,
        "OLf": dac["OLf"][mode_idx] / 2**16,
        "PL1": dac["PL1"][mode_idx] / 2**16,
    }

    ni = {
        key: i_fs_common_a * turns[key] * i_norm[key]
        for key in i_norm.keys()
    }

    ni_il_sum = ni["IL1"] + ni["IL2"] + ni["IL3"]
    ni_op_sum = ni["OLf"] + ni["PL1"]
    delta_ni = ni_op_sum - ni_il_sum

    return {
        "i_norm": i_norm,
        "ni": ni,
        "ni_il_sum_at": ni_il_sum,
        "ni_op_sum_at": ni_op_sum,
        "delta_ni_at": delta_ni,
        "delta_psi_deg": compute_rotation_from_delta_ni(delta_ni),
    }


ni_terms = compute_ni_balance_terms(
    dac=dac,
    mode_idx=mode_idx,
    turns=N_TURNS,
    i_fs_common_a=I_fs_common_A,
)

Gc_phys = {
    "IL1": G_c_1_opt / (N_TURNS["IL1"] * I_fs_common_A)**2,
    "IL2": G_c_2_opt / (N_TURNS["IL2"] * I_fs_common_A)**2,
    "IL3": G_c_3_opt / (N_TURNS["IL3"] * I_fs_common_A)**2,
}

print("NI-balance pre-trim summary:")
print(f"  turns={N_TURNS}")
print(f"  I_fs_common={I_fs_common_A:.3f} A")
print(
    f"  delta_NI [AT] min/max/rms="
    f"{np.min(ni_terms['delta_ni_at']):.2f}/"
    f"{np.max(ni_terms['delta_ni_at']):.2f}/"
    f"{np.sqrt(np.mean(ni_terms['delta_ni_at']**2)):.2f}"
)
print(
    f"  deltaPsi [deg] min/max/rms="
    f"{np.min(ni_terms['delta_psi_deg']):.3f}/"
    f"{np.max(ni_terms['delta_psi_deg']):.3f}/"
    f"{np.sqrt(np.mean(ni_terms['delta_psi_deg']**2)):.3f}"
)
print(
    f"  G_c physical [1/(AT^2*m)] IL1={Gc_phys['IL1']:.3e}, "
    f"IL2={Gc_phys['IL2']:.3e}, IL3={Gc_phys['IL3']:.3e}"
)

print("\nCompact NI table:")
print(f"{'Mag':>8s} | {'NI_IL_sum':>11s} | {'NI_OP_sum':>11s} | {'dNI [AT]':>10s} | {'dPsi [deg]':>10s}")
print("-" * 63)
for mag, il_sum, op_sum, dni, dpsi in zip(
    mode_mag,
    ni_terms["ni_il_sum_at"],
    ni_terms["ni_op_sum_at"],
    ni_terms["delta_ni_at"],
    ni_terms["delta_psi_deg"],
):
    print(f"{mag:8.0f} | {il_sum:11.2f} | {op_sum:11.2f} | {dni:10.2f} | {dpsi:10.3f}")


NI-balance pre-trim summary:
  turns={'IL1': 2950, 'IL2': 2050, 'IL3': 2000, 'OLf': 2000, 'PL1': 2650}
  I_fs_common=2.000 A
  delta_NI [AT] min/max/rms=-112.54/80.69/50.87
  deltaPsi [deg] min/max/rms=-3.417/2.450/1.545
  G_c physical [1/(AT^2*m)] IL1=1.001e-05, IL2=1.873e-05, IL3=6.419e-06

Compact NI table:
     Mag |   NI_IL_sum |   NI_OP_sum |   dNI [AT] | dPsi [deg]
---------------------------------------------------------------
   30000 |     7648.02 |     7567.02 |     -81.00 |     -2.460
   40000 |     7486.33 |     7567.02 |      80.69 |      2.450
   50000 |     7536.59 |     7567.02 |      30.43 |      0.924
   60000 |     7565.28 |     7567.02 |       1.73 |      0.053
   80000 |     7583.03 |     7567.02 |     -16.02 |     -0.486
  100000 |     7659.76 |     7567.02 |     -92.75 |     -2.816
  120000 |     7569.50 |     7567.02 |      -2.48 |     -0.075
  150000 |     7566.85 |     7567.02 |       0.16 |      0.005
  200000 |     7679.56 |     7567.02 |    -112.54 |     -